In [3]:
import json
import networkx as nx
from datetime import datetime, timedelta
from collections import defaultdict, deque

def get_fingerprint(alert: dict) -> str:
    """
    Tạo định danh duy nhất cho loại alert dựa trên luật ở Section 2.1.
    Bỏ qua timestamp và value vì chúng thay đổi liên tục.
    """
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"

In [15]:
# 1. Đọc và xây dựng Service Graph
def build_graph(services_json_path: str) -> nx.DiGraph:
    g = nx.DiGraph()
    with open(services_json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    for svc in data['services']:
        g.add_node(svc['name'], type='service')
    for store in data['stores']:
        g.add_node(store['name'], type='store')
    for edge in data['edges']:
        g.add_edge(edge['from'], edge['to'], type=edge['type'])
    return g

graph = build_graph("./dataset/services.json")

# 2. Đọc danh sách Alerts
alerts = []
with open("./dataset/alerts_sample.jsonl", 'r') as f:
    for line in f:
        if line.strip():
            alerts.append(json.loads(line))


In [16]:
def session_groups(alerts: list[dict], gap_sec: int = 120) -> list[list[dict]]:
    if not alerts:
        return []
    
    # Sắp xếp các alert theo trình tự thời gian tăng dần
    sorted_alerts = sorted(alerts, key=lambda a: a['ts'])
    groups = [[sorted_alerts[0]]]
    
    for alert in sorted_alerts[1:]:
        ts = datetime.fromisoformat(alert['ts'].replace('Z', '+00:00'))
        last_ts = datetime.fromisoformat(groups[-1][-1]['ts'].replace('Z', '+00:00'))
        
        if (ts - last_ts).total_seconds() <= gap_sec:
            groups[-1].append(alert)
        else:
            groups.append([alert])
    
    return groups

In [17]:
def topology_group(alerts: list[dict], graph: nx.DiGraph, max_hop: int = 1) -> list[list[dict]]:
    if not alerts:
        return []
    
    undirected = graph.to_undirected()
    by_service = defaultdict(list)
    for a in alerts:
        by_service[a['service']].append(a)
        
    services_with_alerts = list(by_service.keys())
    
    # Thuật toán Union-Find để gom nhóm các service có khoảng cách ngắn
    parent = {s: s for s in services_with_alerts}
    
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    
    def union(x, y):
        parent[find(x)] = find(y)
        
    for i, s1 in enumerate(services_with_alerts):
        for s2 in services_with_alerts[i+1:]:
            try:
                # Kiểm tra độ dài đường đi ngắn nhất giữa 2 node dịch vụ
                dist = nx.shortest_path_length(undirected, s1, s2)
                if dist <= max_hop:
                    union(s1, s2)
            except nx.NetworkXNoPath:
                continue
                
    groups_dict = defaultdict(list)
    for s in services_with_alerts:
        groups_dict[find(s)].extend(by_service[s])
        
    return list(groups_dict.values())

In [34]:
def correlate_pipeline(alerts: list[dict], graph: nx.DiGraph, gap_sec: int = 120, max_hop: int = 2) -> dict:
    sessions = session_groups(alerts, gap_sec=gap_sec)
    output_clusters = []
    
    for session_idx, session_alerts in enumerate(sessions):
        topo_groups = topology_group(session_alerts, graph, max_hop=max_hop)
        
        for group_idx, group in enumerate(topo_groups):
            fps = sorted(list(set(get_fingerprint(a) for a in group)))
            services = sorted(list(set(a['service'] for a in group)))
            alert_ids = [a['id'] for a in group]
            severities = [a['severity'] for a in group]
            
            # Tính mức độ nghiêm trọng cao nhất trong cụm: crit > warn
            max_severity = "crit" if "crit" in severities else "warn"
            
            output_clusters.append({
                'cluster_id': f'c-{session_idx:03d}-{group_idx:03d}',
                'alert_count': len(group),
                'services': services,
                'alert_ids': alert_ids,
                'time_range': [min(a['ts'] for a in group), max(a['ts'] for a in group)],
                'max_severity': max_severity,
                'fingerprints': fps
            })
            
    reduction_ratio = 1.0 - (len(output_clusters) / len(alerts)) if alerts else 0.0
    
    return {
        "input_alerts": len(alerts),
        "output_clusters": len(output_clusters),
        "reduction_ratio": round(reduction_ratio, 2),
        "clusters": output_clusters
    }

# Thực thi pipeline và lưu file kết quả
result_summary = correlate_pipeline(alerts, graph, gap_sec=30, max_hop=2)

import os
os.makedirs("results", exist_ok=True)
with open("results/cluster_summary.json", "w") as f:
    json.dump(result_summary, f, indent=2)
print(f"Pipeline hoàn tất! Tỷ lệ giảm tải alert: {result_summary['reduction_ratio'] * 100}%")

Pipeline hoàn tất! Tỷ lệ giảm tải alert: 75.0%
